# 🎹 REAL MAESTRO — One-to-Many Music Generation (12 Pitch Classes)
## LSTM vs GRU vs TCN | Deep Learning Evaluation

**Real data:** Official **MAESTRO v3 MIDI** piano performances, *not synthetic music*. Each sample starts from **one actual pitch class** (C through B); the model generates **32 following pitch classes**.

**Important research honesty:** This is a **12-class pitch-class task**, not the original 88-key piano-pitch task. Octave information is deliberately discarded (`MIDI_pitch % 12`). A higher score is not directly comparable to 88-class accuracy. **70%+ accuracy is a goal, not a guarantee**, and no results are pre-filled. Train/validation/test splits follow the official *performance-level* split; results are measured only after training.

**Two evaluation regimes:** (1) teacher-forced note-prediction metrics using previous **true** notes; (2) free-running One-to-Many generation from **one seed**, where the model feeds back its own predictions. Teacher-forced accuracy must **not** be described as free-generation accuracy.

### Run in Google Colab
1. **Runtime → Change runtime type → T4 GPU** (if available).
2. **Runtime → Run all**. The real MIDI dataset downloads automatically; training can take a while depending on Colab resources.
3. See the **final results dashboard, confusion matrices, and playable audio** at the bottom.


## 1 | Setup and reproducibility

In [ ]:
%pip -q install pretty_midi

import os, time, random, copy, zipfile, urllib.request, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Audio, Markdown
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import pretty_midi
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix, ConfusionMatrixDisplay

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__, '| Device:', DEVICE)
if DEVICE.type == 'cuda': print('GPU:', torch.cuda.get_device_name(0))

## 2 | Download verified real MAESTRO v3 MIDI performances

In [ ]:
URL = 'https://storage.googleapis.com/magentadata/datasets/maestro/v3.0.0/maestro-v3.0.0-midi.zip'
EXPECTED_SHA256 = '70470ee253295c8d2c71e6d9d4a815189e35c89624b76d22fce5a019d5dde12c'
ROOT = Path('/content/maestro_data') if Path('/content').exists() else Path('./maestro_data')
ROOT.mkdir(parents=True, exist_ok=True)
ZIP_PATH = ROOT / 'maestro-v3.0.0-midi.zip'

if not ZIP_PATH.exists():
    print('Downloading official MAESTRO MIDI archive (~56 MB)...')
    urllib.request.urlretrieve(URL, ZIP_PATH)
else:
    print('Using existing archive:', ZIP_PATH)

sha = hashlib.sha256()
with open(ZIP_PATH, 'rb') as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b''):
        sha.update(chunk)
if sha.hexdigest() != EXPECTED_SHA256:
    ZIP_PATH.unlink(missing_ok=True)
    raise RuntimeError('Dataset SHA256 verification failed. Archive removed; rerun to download again.')
print('Official dataset SHA256: VERIFIED')

OUT = ROOT / 'extracted'
OUT.mkdir(exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    # Standard MAESTRO archive contains the maestro-v3.0.0 root directory
    if not list(OUT.rglob('maestro-v3.0.0.csv')):
        print('Extracting MIDI files...')
        z.extractall(OUT)
metadata_paths = list(OUT.rglob('maestro-v3.0.0.csv'))
assert len(metadata_paths) == 1, 'Cannot locate MAESTRO metadata after extraction.'
BASE = metadata_paths[0].parent
meta = pd.read_csv(metadata_paths[0])
assert {'split', 'midi_filename', 'canonical_composer'}.issubset(meta.columns)
print('Official files:', len(meta))
print(meta['split'].value_counts().to_string())
print('Sample:', meta[['canonical_composer','canonical_title','split']].iloc[0].to_dict())

## 3 | Build a genuine 12-pitch-class dataset
MIDI pitches across octaves are mapped to **12 pitch classes** using `% 12` (C=0, C♯=1, …, B=11). The input for each example is **one observed pitch class**; the label is the following **32 observed** pitch classes. MIDI notes are ordered by onset time then pitch; this is a simplified monophonic tokenization of polyphonic piano music. **No arbitrary category labels, no generated targets.**

The split is made by **performance/file**, not by random windows, preventing the same performance appearing in train and test. A fixed subset of official split files is selected for Colab feasibility.

In [ ]:
SEQ_LEN = 32
MIN_PITCH, MAX_PITCH = 21, 108  # filter for real piano pitches before reducing to pitch classes
VOCAB = 12  # C, C#, D, ..., B (octave discarded)
EMBED = 64
HIDDEN = 128
BATCH_SIZE = 96
EPOCHS = 25
LR = 0.001

# Colab-friendly subset. Set to None to process all performances (slower).
MAX_FILES = {'train': 400, 'validation': 60, 'test': 60}
WINDOWS_PER_FILE = 64

def midi_pitch_stream(path):
    midi = pretty_midi.PrettyMIDI(str(path))
    notes = []
    for instrument in midi.instruments:
        if instrument.is_drum: continue
        for note in instrument.notes:
            if MIN_PITCH <= note.pitch <= MAX_PITCH:
                notes.append((note.start, note.pitch % 12))
    # Sort by onset, then pitch for deterministic ordering of simultaneous notes
    notes.sort(key=lambda z: (z[0], z[1]))
    return np.asarray([p for _, p in notes], dtype=np.int64)

def extract_split(split):
    rows = meta[meta['split'] == split].sort_values('midi_filename')
    if MAX_FILES[split] is not None:
        rows = rows.head(MAX_FILES[split])
    starts, continuations, file_ids = [], [], []
    failures = 0
    for item in rows.itertuples(index=False):
        filepath = BASE / item.midi_filename
        try:
            pitches = midi_pitch_stream(filepath)
        except Exception as exc:
            failures += 1
            continue
        possible = len(pitches) - SEQ_LEN
        if possible <= 0: continue
        starts_at = np.unique(np.linspace(0, possible-1, num=min(WINDOWS_PER_FILE, possible), dtype=int))
        for pos in starts_at:
            starts.append(pitches[pos])
            continuations.append(pitches[pos+1: pos+1+SEQ_LEN])
            file_ids.append(item.midi_filename)
    X = np.asarray(starts, dtype=np.int64)
    Y = np.asarray(continuations, dtype=np.int64)
    assert len(X) > 0 and Y.shape[1] == SEQ_LEN, f'Empty or malformed {split} split.'
    print(f'{split:10s}: {len(X):5d} real-data windows from {len(set(file_ids)):3d} MIDI performances; parse failures={failures}')
    return X, Y, set(file_ids)

X_train, Y_train, train_files = extract_split('train')
X_val, Y_val, val_files = extract_split('validation')
X_test, Y_test, test_files = extract_split('test')
assert train_files.isdisjoint(val_files | test_files)
assert val_files.isdisjoint(test_files)
assert Y_train.min() >= 0 and Y_train.max() < VOCAB
PITCH_NAMES = ['C','C#','D','D#','E','F','F#','G','G#','A','A#','B']
print('\nOne input pitch class:', PITCH_NAMES[int(X_train[0])])
print('Target: next', SEQ_LEN, 'real pitch classes:', [PITCH_NAMES[p] for p in Y_train[0][:12]], '...')

fig, ax = plt.subplots(figsize=(12, 3.5))
ax.step(np.arange(SEQ_LEN+1), np.r_[X_train[0],Y_train[0]], where='mid')
ax.set(xlabel='Note-on event index', ylabel='Pitch class (0=C … 11=B)', title='Real MAESTRO pitch classes (one seed + next 32)')
ax.grid(alpha=.25)
plt.show()

## 4 | DataLoaders and repeat-previous-note baseline

In [ ]:
def make_loader(x, y, shuffle=False):
    return DataLoader(TensorDataset(torch.from_numpy(x), torch.from_numpy(y)),
                      batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=0)
train_loader = make_loader(X_train, Y_train, True)
val_loader = make_loader(X_val, Y_val)
test_loader = make_loader(X_test, Y_test)

# Under teacher forcing, a very simple baseline is to repeat the previous true note.
prev_test = np.concatenate([X_test[:, None], Y_test[:, :-1]], axis=1)
repeat_accuracy = (prev_test == Y_test).mean() * 100
print(f'Repeat-previous-note teacher-forced test accuracy baseline: {repeat_accuracy:.2f}%')

## 5 | Three **causal autoregressive** sequence decoders
LSTM and GRU use 2 recurrent layers; TCN uses four residual causal-convolution blocks with dilations 1, 2, 4, and 8. The architectures use the same pitch-class input/target vocabulary and same splits. Their parameter counts differ; compare accuracy **and** efficiency.

At training time, the decoder sees previous **ground-truth** notes (teacher forcing). At generation time, it sees only the single starting note plus its own outputs. This is legitimate autoregressive One-to-Many generation.

In [ ]:
class RecurrentDecoder(nn.Module):
    def __init__(self, cell='LSTM'):
        super().__init__()
        self.pitch_embed = nn.Embedding(VOCAB, EMBED)
        self.seed_embed = nn.Embedding(VOCAB, EMBED)
        klass = nn.LSTM if cell == 'LSTM' else nn.GRU
        self.rnn = klass(EMBED * 2, HIDDEN, num_layers=2, batch_first=True)
        self.out = nn.Linear(HIDDEN, VOCAB)

    def forward(self, seed, prev_notes):
        # seed: [B]; prev_notes: [B,T] (seed then true/generated preceding tokens)
        p = self.pitch_embed(prev_notes)
        s = self.seed_embed(seed)[:,None,:].expand(-1, prev_notes.size(1), -1)
        h, _ = self.rnn(torch.cat([p,s], dim=-1))
        return self.out(h)  # [B,T,12]

class CausalConv(nn.Module):
    def __init__(self, in_channels, out_channels, dilation):
        super().__init__()
        self.pad = 2 * dilation   # kernel_size=3
        self.conv = nn.Conv1d(in_channels, out_channels, 3,
                              dilation=dilation, padding=self.pad)
    def forward(self,x):
        y = self.conv(x)
        return y[:,:,:-self.pad] if self.pad else y

class TCNBlock(nn.Module):
    def __init__(self, inc, outc, dilation):
        super().__init__()
        self.main = nn.Sequential(CausalConv(inc,outc,dilation), nn.ReLU(),
                                  nn.Dropout(0.1), CausalConv(outc,outc,dilation),
                                  nn.ReLU(), nn.Dropout(0.1))
        self.skip = nn.Conv1d(inc,outc,1) if inc != outc else nn.Identity()
    def forward(self,x): return torch.relu(self.main(x) + self.skip(x))

class TCNDecoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.pitch_embed = nn.Embedding(VOCAB, EMBED)
        self.seed_embed = nn.Embedding(VOCAB, EMBED)
        self.stack = nn.Sequential(TCNBlock(EMBED*2,HIDDEN,1),
                                   TCNBlock(HIDDEN,HIDDEN,2),
                                   TCNBlock(HIDDEN,HIDDEN,4),
                                   TCNBlock(HIDDEN,HIDDEN,8))
        self.out = nn.Linear(HIDDEN,VOCAB)
    def forward(self, seed, prev_notes):
        p = self.pitch_embed(prev_notes)
        s = self.seed_embed(seed)[:,None,:].expand(-1,prev_notes.size(1),-1)
        z = self.stack(torch.cat([p,s],-1).transpose(1,2)).transpose(1,2)
        return self.out(z)

def make_models():
    return {'LSTM': RecurrentDecoder('LSTM').to(DEVICE),
            'GRU': RecurrentDecoder('GRU').to(DEVICE),
            'TCN': TCNDecoder().to(DEVICE)}
models = make_models()
with torch.no_grad():
    demo_seed = torch.tensor([0,7], device=DEVICE)
    demo_prev = demo_seed[:,None].expand(2,SEQ_LEN)
    for name,m in models.items():
        m.eval()
        shape = tuple(m(demo_seed,demo_prev).shape)
        params = sum(p.numel() for p in m.parameters())
        print(name, 'output:', shape, 'parameters:', f'{params:,}')
        assert shape == (2, SEQ_LEN, VOCAB)

## 6 | Train each model with validation checkpoint selection (25 epochs)

In [ ]:
loss_fn = nn.CrossEntropyLoss()

def step_inputs(seed, target):
    # Teacher forcing: previous true note is visible at every prediction position.
    return torch.cat([seed[:,None], target[:,:-1]], dim=1)

def evaluate(model, loader):
    model.eval()
    total_ce, correct, top5, count = 0., 0, 0, 0
    with torch.inference_mode():
        for seed, target in loader:
            seed, target = seed.to(DEVICE), target.to(DEVICE)
            logits = model(seed, step_inputs(seed,target))
            total_ce += loss_fn(logits.reshape(-1,VOCAB), target.reshape(-1)).item()*target.numel()
            correct += (logits.argmax(-1)==target).sum().item()
            top5 += (logits.topk(5,dim=-1).indices==target.unsqueeze(-1)).any(-1).sum().item()
            count += target.numel()
    return {'loss':total_ce/count, 'accuracy':100*correct/count, 'top5':100*top5/count}

def fit(model, name):
    optim = torch.optim.Adam(model.parameters(),lr=LR)
    hist=[]; best_loss=float('inf'); best=None
    if DEVICE.type=='cuda': torch.cuda.synchronize()
    t0=time.perf_counter()
    for ep in range(1,EPOCHS+1):
        model.train(); total=0.; n=0
        for seed,target in train_loader:
            seed,target=seed.to(DEVICE),target.to(DEVICE)
            optim.zero_grad(set_to_none=True)
            logits = model(seed,step_inputs(seed,target))
            loss = loss_fn(logits.reshape(-1,VOCAB),target.reshape(-1))
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optim.step()
            total += loss.item()*target.numel(); n += target.numel()
        val = evaluate(model,val_loader)
        hist.append({'epoch':ep,'train_loss':total/n,
                     'val_loss':val['loss'],'val_accuracy':val['accuracy']})
        if val['loss'] < best_loss:
            best_loss=val['loss']
            best={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        print(f'{name:4s} epoch {ep:02d}/{EPOCHS}: train_loss={total/n:.3f} '
              f'val_loss={val["loss"]:.3f} val_acc={val["accuracy"]:.2f}%')
    if DEVICE.type=='cuda': torch.cuda.synchronize()
    elapsed=time.perf_counter()-t0
    model.load_state_dict(best)
    return pd.DataFrame(hist),elapsed

histories={}; training_times={}
for name,model in models.items():
    print('\nTRAINING',name)
    h,elapsed=fit(model,name)
    histories[name]=h
    training_times[name]=elapsed
    print(f'{name} total training+validation time: {elapsed:.1f}s')
print('\nFinished training and restored each model’s best-validation-loss checkpoint.')

## 7 | Held-out test evaluation: accuracy, F1, perplexity, exact match, timing
These are **teacher-forced next-pitch-class prediction** results. The 32-step *exact-sequence accuracy* counts a window correct only when **all 32** individual teacher-forced predictions match. It is usually much lower than note accuracy.

The repeat-previous-note baseline is printed alongside the trained models to help interpret performance.

In [ ]:
rows = []
all_preds, all_true = {}, {}
for name, model in models.items():
    test = evaluate(model, test_loader)
    preds, gold = [], []
    model.eval()
    sample_batch = next(iter(test_loader))[0].to(DEVICE)
    fake_prev = sample_batch[:, None].expand(-1, SEQ_LEN)
    with torch.inference_mode():
        for _ in range(10): model(sample_batch, fake_prev)
        if DEVICE.type == 'cuda': torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(30): model(sample_batch, fake_prev)
        if DEVICE.type == 'cuda': torch.cuda.synchronize()
        ms = 1000 * (time.perf_counter() - t0) / 30
        for seed, target in test_loader:
            seed, target = seed.to(DEVICE), target.to(DEVICE)
            pred = model(seed, step_inputs(seed, target)).argmax(-1)
            preds.append(pred.cpu().numpy())
            gold.append(target.cpu().numpy())
    pred, gt = np.concatenate(preds), np.concatenate(gold)
    all_preds[name], all_true[name] = pred, gt
    p, y = pred.ravel(), gt.ravel()
    rows.append({
        'Model': name,
        'Top-1 Accuracy (%)': test['accuracy'],
        'Top-5 Accuracy (%)': test['top5'],
        'Weighted Precision (%)': 100*precision_score(y,p,average='weighted',zero_division=0),
        'Weighted Recall (%)': 100*recall_score(y,p,average='weighted',zero_division=0),
        'Weighted F1 (%)': 100*f1_score(y,p,average='weighted',zero_division=0),
        'Macro F1 (%)': 100*f1_score(y,p,labels=np.arange(VOCAB),average='macro',zero_division=0),
        'Exact 32-note Match (%)': 100*np.mean(np.all(pred == gt,axis=1)),
        'Test Cross-Entropy': test['loss'],
        'Perplexity': float(np.exp(min(test['loss'],50))),
        'Parameters': sum(z.numel() for z in model.parameters()),
        'Train+Validation (s)': training_times[name],
        'Forward Latency (ms/batch)': ms,
    })
results = pd.DataFrame(rows).sort_values('Top-1 Accuracy (%)', ascending=False).reset_index(drop=True)
print(f'\nRepeat-previous-note baseline: {repeat_accuracy:.2f}% (teacher-forced)')
display(results.round(3))
results.to_csv('MAESTRO_12class_LSTM_GRU_TCN_results.csv',index=False)


## 8 | Presentation-ready comparison dashboard

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
for name, h in histories.items():
    axes[0,0].plot(h['epoch'],h['train_loss'],label=name)
    axes[0,1].plot(h['epoch'],h['val_loss'],label=name)
    axes[1,0].plot(h['epoch'],h['val_accuracy'],label=name)
axes[0,0].set(title='Training cross-entropy',xlabel='Epoch',ylabel='Loss')
axes[0,1].set(title='Validation cross-entropy',xlabel='Epoch',ylabel='Loss')
axes[1,0].set(title='Validation note accuracy',xlabel='Epoch',ylabel='Accuracy (%)')
axes[1,0].axhline(repeat_accuracy,linestyle=':',color='gray',label='Test repeat baseline (reference only)')
axes[1,1].bar(results['Model'],results['Top-1 Accuracy (%)'])
axes[1,1].set(title='Held-out test Top-1 accuracy',ylabel='Accuracy (%)',ylim=(0,100))
for ax in axes.flat:ax.grid(alpha=.25)
for ax in axes.flat[:3]:ax.legend()
plt.suptitle('Real MAESTRO / 12-class One-to-Many: LSTM vs GRU vs TCN',fontweight='bold')
plt.tight_layout()
plt.savefig('MAESTRO_12class_dashboard.png',dpi=200,bbox_inches='tight')
plt.show()

fig,axes=plt.subplots(1,3,figsize=(16,4))
for ax, col in zip(axes,['Weighted F1 (%)','Exact 32-note Match (%)','Train+Validation (s)']):
    ax.bar(results['Model'],results[col]);ax.set_title(col);ax.grid(axis='y',alpha=.2)
plt.tight_layout();plt.show()


## 9 | Twelve-class confusion matrices (test set)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(19, 5.5))
for ax, name in zip(axes, models):
    cm = confusion_matrix(all_true[name].ravel(),all_preds[name].ravel(),labels=np.arange(VOCAB),normalize='true')
    cm = np.nan_to_num(cm)
    im=ax.imshow(cm,vmin=0,vmax=1,cmap='Blues')
    ax.set(title=f'{name}: normalized confusion',xlabel='Predicted',ylabel='Actual',xticks=np.arange(12),yticks=np.arange(12))
    ax.set_xticklabels(PITCH_NAMES,rotation=90);ax.set_yticklabels(PITCH_NAMES)
fig.colorbar(im,ax=axes.ravel().tolist(),shrink=.75,label='Proportion')
plt.savefig('MAESTRO_12class_confusion.png',dpi=180,bbox_inches='tight')
plt.show()


## 10 | Generate **32 notes from ONE pitch class** (no teacher forcing)
At inference, the seed is a **pitch class (0–11)**. The model is allowed to use its **own previously predicted notes** and receives no continuation from MAESTRO. We synthesize the output in **one fixed octave** for demonstration because pitch-class modeling discards octave information. This is a limitation of the simplified 12-class task.

In [ ]:
@torch.inference_mode()
def generate(model, seed_pc=0, n_notes=SEQ_LEN, temperature=0.9, top_k=5, sample=True, rng_seed=7):
    assert 0 <= seed_pc < VOCAB and temperature > 0
    model.eval()
    seed = torch.tensor([seed_pc],device=DEVICE,dtype=torch.long)
    prev = seed[:,None]
    generated = []
    generator = torch.Generator(device=DEVICE.type).manual_seed(rng_seed)
    for _ in range(n_notes):
        logits=model(seed,prev)[:,-1,:]/temperature
        if sample:
            vals,idx = logits.topk(min(top_k,VOCAB),dim=-1)
            probs = torch.softmax(vals,dim=-1)
            choice = torch.multinomial(probs,1,generator=generator)
            token = idx.gather(1,choice)
        else:
            token = logits.argmax(-1,keepdim=True)
        generated.append(int(token.item()))
        prev=torch.cat([prev,token],dim=1)
    return generated

SEED_PC = 0 # C pitch class, one input for all three models
samples = {name:generate(model,SEED_PC) for name,model in models.items()}
print('ONE INPUT:',PITCH_NAMES[SEED_PC], '(pitch class ID', SEED_PC, ')')
for name,seq in samples.items():
    print(f'{name:4s}:', ' '.join(PITCH_NAMES[p] for p in seq))
fig,ax=plt.subplots(figsize=(14,4))
for name,seq in samples.items():ax.plot(np.arange(1,SEQ_LEN+1),seq,marker='.',label=name)
ax.set(title='Free-running generation: one seed → 32 pitch classes',xlabel='Output position',ylabel='Predicted pitch class',yticks=list(range(12)))
ax.set_yticklabels(PITCH_NAMES);ax.grid(alpha=.2);ax.legend()
plt.tight_layout();plt.savefig('MAESTRO_12class_generated.png',dpi=180);plt.show()


## 11 | Play generated melody (synthesized demo audio; fixed octave)

In [ ]:
def make_audio(pitch_classes, sec_per_note=0.18, sr=22050):
    parts=[]
    for pc in pitch_classes:
        midi_note=60+int(pc) # C4 through B4: fixed octave by design
        f=pretty_midi.note_number_to_hz(midi_note)
        t=np.arange(int(sr*sec_per_note))/sr
        tone=np.sin(2*np.pi*f*t)+0.2*np.sin(4*np.pi*f*t)
        env=(1-np.exp(-t*65))*np.exp(-t*7)
        parts.append((tone*env).astype(np.float32))
    wave=np.concatenate(parts)
    wave/=max(1.,np.max(np.abs(wave)))
    return wave,sr
for name,seq in samples.items():
    display(Markdown(f'**{name}: one C pitch-class seed → 32 predicted pitch classes**'))
    wave,sr=make_audio([SEED_PC]+seq)
    display(Audio(wave,rate=sr))


## 12 | Interactive live demonstration for evaluation

In [ ]:
import ipywidgets as widgets
from IPython.display import clear_output
pc_widget=widgets.Dropdown(options=[(name,i) for i,name in enumerate(PITCH_NAMES)],value=0,description='One input')
model_widget=widgets.Dropdown(options=list(models),value='LSTM',description='Decoder')
random_widget=widgets.IntSlider(value=7,min=0,max=100,description='Sample seed')
button=widgets.Button(description='Generate 32 notes',button_style='success')
output_box=widgets.Output()
def on_generate(_):
    with output_box:
        clear_output(wait=True)
        notes=generate(models[model_widget.value],pc_widget.value,rng_seed=random_widget.value)
        print('One pitch-class input:',PITCH_NAMES[pc_widget.value])
        print('Decoder:',model_widget.value,'| 32 output pitch classes')
        print('Generated:', ' '.join(PITCH_NAMES[n] for n in notes))
        audio,sr=make_audio([pc_widget.value]+notes)
        display(Audio(audio,rate=sr))
button.on_click(on_generate)
display(widgets.VBox([pc_widget,model_widget,random_widget,button,output_box]))


## 13 | Final presentation table, ranking, and files

In [ ]:
display(Markdown('### FINAL REAL-DATA OBSERVATION TABLE'))
display(results.round(3))
max_acc=results['Top-1 Accuracy (%)'].max()
winners=results.loc[np.isclose(results['Top-1 Accuracy (%)'],max_acc),'Model'].tolist()
print('Highest teacher-forced Top-1 test accuracy:',round(max_acc,2),'%')
print('Top model(s):',', '.join(winners))
print('Repeat-previous-note baseline:',round(repeat_accuracy,2),'%')
if max_acc >= 70:
    print('Target met: ≥70% teacher-forced accuracy. This is not free-running accuracy.')
else:
    print('Below 70%: report the real result. Investigate validation curves, baseline, and data rather than changing the metric.')
for name,model in models.items():
    torch.save(model.state_dict(),f'MAESTRO_12class_{name}_decoder.pth')
print('Saved CSV, dashboard, confusion matrix, generated-note plot, and trained model weights.')


## 14 | Viva-ready interpretation
**Problem statement:** Compare **LSTM, GRU, TCN** as autoregressive decoders for **One-to-Many real-piano pitch-class generation**.

**Real dataset:** MAESTRO v3 MIDI. **Input:** one observed pitch class (0–11). **Output:** 32 subsequent real pitch classes during training; 32 freely generated pitch classes at inference.

**Why a decoder?** Each model predicts the next pitch class conditioned on the input seed and preceding pitch classes. During training, preceding true notes speed learning; during generation, the model feeds back its own predictions.

**Why 12 classes?** Pitch classes merge octaves, making the target simpler while keeping actual MAESTRO note transitions. **Tradeoff:** no register or octave prediction, so generated demo audio uses a fixed octave. This is **not equivalent to 88-key piano transcription**.

**Why could accuracy be low?** An observed starting pitch class does not uniquely determine a 32-note continuation; there may be many valid real-music sequences. Note-level Top-1 metrics score only the recorded next note, not music quality.

**Teacher-forced Top-1** should not be described as the percentage of good melodies. The **audio demo** is a separate qualitative result. Report actual accuracy and baselines without altering them to meet a threshold.
